# MLSR Binning Demo

This notebook uses the reusable `estimate_binning_scales` function from `binning.py`. It estimates the finite-difference/bin width `h_fd` and Gaussian/RBF kernel length `lambda_opt` from a 1D scattering curve.

In [ ]:
import matplotlib.pyplot as plt

from binning import synthetic_scattering_data, estimate_binning_scales, rbf_gpr_predict

## Synthetic Data

The helper below creates a small noisy scattering curve so the example is fully reproducible.

In [ ]:
total_counts = 1.0e4
q, intensity, intensity_error, true_intensity = synthetic_scattering_data(
    n_points=201,
    total_counts=total_counts,
    seed=7,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(q, intensity, yerr=intensity_error, fmt=".", ms=3, alpha=0.45, label="noisy data")
ax.plot(q, true_intensity, lw=2, label="truth used for simulation")
ax.set_xlabel("Q")
ax.set_ylabel("I(Q)")
ax.legend(frameon=False)
fig.tight_layout()

## Estimate Best Bin Width and Kernel Size

In [ ]:
result = estimate_binning_scales(
    q,
    intensity,
    total_counts=total_counts,
    intensity_error=intensity_error,
    window_frac=0.15,
    polyorder=3,
)

print(f"Median data bin width: {result.data_bin_width:.6e}")
print(f"Best finite-difference/bin width h_fd: {result.h_fd:.6e}")
print(f"Best Gaussian/RBF kernel size lambda_opt: {result.lambda_opt:.6e}")
print()
print(f"alpha: {result.alpha:.6e}")
print(f"beta:  {result.beta:.6e}")
print(f"gamma: {result.gamma:.6e}")
print(f"chi:   {result.chi:.6e}")
print(f"Savitzky-Golay window used: {result.window_length}")

## GPR Reconstruction with Optimized Kernel

Use `lambda_opt` as the RBF kernel size and compare the posterior mean with the observation and ground truth.

In [ ]:
gpr_intensity, gpr_std = rbf_gpr_predict(
    q,
    intensity,
    intensity_error,
    q_predict=q,
    kernel_size=result.lambda_opt,
    return_std=True,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(
    q,
    gpr_intensity - gpr_std,
    gpr_intensity + gpr_std,
    color="tab:red",
    alpha=0.18,
    linewidth=0,
    label="GPR +/- 1 sigma",
)
ax.errorbar(q, intensity, yerr=intensity_error, fmt=".", ms=3, alpha=0.35, label="observation")
ax.plot(q, true_intensity, lw=2.2, color="black", label="ground truth")
ax.plot(q, gpr_intensity, lw=2.0, color="tab:red", label="GPR, optimized kernel")
ax.set_xlabel("Q")
ax.set_ylabel("I(Q)")
ax.set_title(f"h_fd = {result.h_fd:.2e}, lambda_opt = {result.lambda_opt:.2e}")
ax.legend(frameon=False)
fig.tight_layout()

## Use Your Own Data

Replace `q_user`, `i_user`, and either `total_counts_user` or `i_err_user` with arrays from your measurement.

In [ ]:
# q_user = ...
# i_user = ...
# total_counts_user = ...
# i_err_user = ...

# estimate = estimate_binning_scales(
#     q_user,
#     i_user,
#     total_counts=total_counts_user,
#     # intensity_error=i_err_user,  # needed only if total_counts is omitted
# )
# estimate.h_fd, estimate.lambda_opt